# 2. MoSDeF Advanced: Custom Design

### What you will do here
- Build a Compound by hand with `add`, `add_bond`, and the geometry methods
- Use ports and `force_overlap` to snap Compounds together like building blocks
- Write your own `Compound` subclasses so molecules become reusable pieces
- Turn a whole system build into a parameterized recipe
- Read SMARTS strings and understand how atom typing decides what an atom is
- Write a small force field XML from scratch and apply it
- Reach into a typed Topology, change a parameter, and write out a screening study

The theme is reproducibility. Every build below is a function of its arguments,
so rerunning it gives you the same system, and changing an argument gives you a
documented variant instead of an edited input file.

In [1]:
import numpy as np

import mbuild as mb
import gmso
from mbuild import Compound
from gmso import ForceField
from gmso.parameterization import apply
from gmso.formats import write_lammpsdata, write_gro, write_top

import logging
from mbuild import mBuildLogger
mBuildLogger().library_logger.setLevel(logging.ERROR)
gmso.gmso_logger.library_logger.setLevel(logging.ERROR)

## mBuild as a set of building blocks

Everything in mBuild is a `Compound`. A single particle is a Compound, a molecule
is a Compound holding particles, and a packed box is a Compound holding
molecules. One container type, nested, so the same methods work at every level.

Three things do most of the work.

- `add` puts a child Compound inside a parent
- `add_bond` records a bond between two particles
- `translate`, `translate_to`, `rotate` and `spin` move a Compound and everything
  inside it

None of this needs PACKMOL or the `path` module. It is placing things by hand,
which is what you want when the arrangement itself is the point.

In [2]:
water = mb.Compound(name="water")

o = mb.Compound(name="O", element="O", pos=[0.0, 0.0, 0.0])
h1 = mb.Compound(name="H", element="H", pos=[0.0757, 0.0586, 0.0])
h2 = mb.Compound(name="H", element="H", pos=[-0.0757, 0.0586, 0.0])

water.add([o, h1, h2])
water.add_bond((o, h1))
water.add_bond((o, h2))

print(water.n_particles, "particles,", water.n_bonds, "bonds")
print("children :", [c.name for c in water.children])
print("particles:", [p.name for p in water.particles()])
water.visualize()

3 particles, 2 bonds
children : ['O', 'H', 'H']
particles: ['O', 'H', 'H']


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

`children` is what you added directly. `particles()` walks the whole tree and
yields the leaves. They agree here because the tree is one level deep, and they
stop agreeing as soon as molecules get nested inside a parent.

Note the explicit `element=`. mBuild will not infer it from the name when you
build a particle this way, and the atom typing later needs it.

In [3]:
a = mb.clone(water)
a.translate([0.5, 0, 0])

rotated = mb.clone(a)
rotated.rotate(np.pi, around=[0, 1, 0])

spun = mb.clone(a)
spun.spin(np.pi, around=[0, 1, 0])

for label, compound in [("start", a), ("rotate", rotated), ("spin", spun)]:
    print(f"{label:<7} center {np.round(compound.center, 3)}")

start   center [0.5   0.039 0.   ]
rotate  center [-0.5    0.039 -0.   ]
spin    center [0.5   0.039 0.   ]


`rotate` turns the Compound about a vector through the origin, so anything
sitting away from the origin swings around it and lands somewhere else. `spin`
turns it about its own center and leaves that center where it was. Reach for
`spin` when you mean orientation, and `rotate` when you mean orientation and
position together.

`translate` takes a displacement, `translate_to` takes a destination for the
center. And `mb.clone` because a Compound lives in one place, so reusing one
means copying it first.

In [4]:
grid = mb.Compound(name="grid")

for i in range(4):
    for j in range(3):
        molecule = mb.clone(water)
        molecule.spin(np.pi / 2 * (i + j), around=[0, 0, 1])
        molecule.translate_to([0.32 * i, 0.32 * j, 0.0])
        grid.add(molecule)

print(grid.n_particles, "particles in", len(list(grid.children)), "molecules")
print("bounding box", np.round(grid.get_boundingbox().lengths, 2), "nm")
grid.visualize()

36 particles in 12 molecules
bounding box [1.11 0.79 0.1 ] nm


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Twelve molecules on a lattice you specified, which is the thing PACKMOL will not
do for you. `mb.fill_box` gives a random arrangement at a density, the `path`
module in the next notebook gives a shape to follow, and placing children
yourself is the third option. It is the right one when you already know where
everything goes.

It is also the mechanism behind the rest of this notebook. The classes below add
children and make bonds exactly like this, they just work out where each piece
lands from ports rather than from coordinates you typed.

## Creating Compound Classes

A `Port` is an open connection point on a Compound. `mb.force_overlap` snaps two
Compounds together at a pair of ports, lining them up and making the bond. That
is the whole idea. Molecules are pieces with studs, and a build is a sequence of
snaps.

Popping a hydrogen off a Compound leaves a `Port` behind automatically, which is
how you turn any molecule into a piece.

To be clear about when this is worth it. For a single toluene, a SMILES string
wins and you should just write one. The payoff comes when you want many related
molecules, because then swapping a substituent is swapping an object instead of
rewriting a string and re-deriving where the ring closure indices land.

In [5]:
class Group(Compound):
    """A compound with one open port, made by popping off one hydrogen."""

    def __init__(self, smiles, name):
        super(Group, self).__init__()
        mb.load(smiles, smiles=True, compound=self)
        self.remove([p for p in self.particles() if p.name == "H"][0])
        self.name = name


class Aryl(Compound):
    """A benzene core with ports opened at the given ring positions, 1 to 6."""

    def __init__(self, positions):
        super(Aryl, self).__init__()
        mb.load("c1ccccc1", smiles=True, compound=self)
        carbons = [p for p in self.particles() if p.name == "C"]
        for position in positions:
            neighbors = self.bond_graph.neighbors(carbons[position - 1])
            self.remove([n for n in neighbors if n.name == "H"][0])

In [6]:
methyl = Group("C", "methyl")
hydroxyl = Group("O", "hydroxyl")
print(methyl.name, methyl.n_particles, "particles,", len(methyl.all_ports()), "port")
print(hydroxyl.name, hydroxyl.n_particles, "particles,", len(hydroxyl.all_ports()), "port")

core = Aryl([1, 4])
print("core", core.n_particles, "particles,", len(core.all_ports()), "ports")
core.visualize(show_ports=True).show()

methyl 4 particles, 1 port
hydroxyl 2 particles, 1 port
core 10 particles, 2 ports


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Snapping pieces together

`decorate` walks the open ports on the core and snaps one group onto each. The
core knows nothing about what a group is, and a group knows nothing about the
core. That separation is what makes them compose.

Note the `mb.clone`. A Compound can only live in one place, so reusing a piece
means cloning it first.

In [7]:
def decorate(positions, groups):
    """Snap one Group onto each open port of an Aryl core."""
    core = Aryl(positions)
    open_ports = core.all_ports()
    for port, group in zip(open_ports, groups):
        piece = mb.clone(group)
        core.add(piece)
        mb.force_overlap(piece, piece.all_ports()[0], port)
    return core


toluene = decorate(positions=[1], groups=[methyl])
print(toluene.n_particles, "particles,", len(toluene.all_ports()), "ports left")
toluene.visualize()

15 particles, 0 ports left


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## The combinatorial payoff

Now the same two classes give you a whole family. Positions and pieces come from
a table, and nothing in the build code changes between rows.

Write the ortho, meta, and para cresol SMILES strings by hand and you have to
track ring closure indices in three different places. Here it is a list of two
integers.

In [11]:
recipes = {
    "toluene":    ([1], [methyl]),
    "phenol":     ([1], [hydroxyl]),
    "o-cresol":   ([1, 2], [methyl, hydroxyl]),
    "m-cresol":   ([1, 3], [methyl, hydroxyl]),
    "p-cresol":   ([1, 4], [methyl, hydroxyl]),
    "mesitylene": ([1, 3, 5], [methyl] * 3),
}

for name, (positions, groups) in recipes.items():
    mol = decorate(positions, groups)
    print(f"{name:<11} positions={str(positions):<10} {mol.n_particles:3d} atoms")
    mol.visualize().show()

toluene     positions=[1]         15 atoms


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

phenol      positions=[1]         13 atoms


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

o-cresol    positions=[1, 2]      16 atoms


...

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

p-cresol    positions=[1, 4]      16 atoms


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

mesitylene  positions=[1, 3, 5]   21 atoms


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Ortho, meta, and para put the two groups at genuinely
different distances, because the ports sit on the ring carbons you asked for.

In [9]:
for label, positions in [("ortho", [1, 2]), ("meta", [1, 3]), ("para", [1, 4])]:
    mol = decorate(positions, [methyl, hydroxyl])
    carbon = [p for p in mol.particles() if p.name == "C"][-1]
    oxygen = [p for p in mol.particles() if p.name == "O"][0]
    print(f"{label:<6} methyl carbon to oxygen: {np.linalg.norm(carbon.pos - oxygen.pos):.3f} nm")

ortho  methyl carbon to oxygen: 0.247 nm
meta   methyl carbon to oxygen: 0.427 nm
para   methyl carbon to oxygen: 0.493 nm


## Pieces can be parameterized too

A `Group` is built from a SMILES string, so a piece can take arguments of its
own. An alkyl tail of length n is a piece, and the core never learns anything
about it.

`decorate([1, 4], [tail, hydroxyl])` is a 4-alkylphenol, the precursor to the
alkylphenol ethoxylate surfactants. So the chain length is a real chemical knob
rather than a demo parameter.

In [10]:
def alkylphenol(n_carbons):
    """A 4-alkylphenol with an n carbon tail."""
    tail = Group("C" * n_carbons, f"C{n_carbons}")
    mol = decorate([1, 4], [tail, hydroxyl])
    mol.name = f"C{n_carbons}_phenol"
    return mol


for n in (4, 8, 12):
    mol = alkylphenol(n)
    print(f"{mol.name:<11} {mol.n_particles:3d} atoms, bounding box "
          f"{np.round(mol.get_boundingbox().lengths, 2)} nm")

alkylphenol(12).visualize()

C4_phenol    25 atoms, bounding box [0.76 0.54 0.3 ] nm
C8_phenol    37 atoms, bounding box [0.7  1.21 0.4 ] nm
C12_phenol   49 atoms, bounding box [1.07 0.71 0.74] nm


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## A recipe for a whole system

The same idea scales up. Instead of one molecule, the class below builds an
entire binary mixture and takes the system-specific knobs as arguments.

Think about what belongs in the signature. Anything you would otherwise change
by hand between runs should be a parameter.

In [ ]:
class AlkylphenolMixture(Compound):
    """A box of two 4-alkylphenol tail lengths at a set mole fraction."""

    def __init__(self, n_short, n_long, n_molecules, long_fraction, box_length):
        super(AlkylphenolMixture, self).__init__()
        n_long_chains = int(round(n_molecules * long_fraction))
        n_short_chains = n_molecules - n_long_chains
        short = alkylphenol(n_short)
        long = alkylphenol(n_long)
        # packmol rejects a component with zero molecules, so drop empties
        components = [(short, n_short_chains), (long, n_long_chains)]
        components = [(c, n) for c, n in components if n > 0]
        self.add(
            mb.fill_box(
                compound=[c for c, _ in components],
                n_compounds=[n for _, n in components],
                box=[box_length] * 3,
            )
        )
        self.name = f"alkylphenols_{n_short}_{n_long}_x{long_fraction}"
        self.composition = {short.name: n_short_chains, long.name: n_long_chains}

In [ ]:
mixture = AlkylphenolMixture(
    n_short=4, n_long=12, n_molecules=40, long_fraction=0.25, box_length=4.5
)
print(mixture.name, mixture.composition)
print(mixture.n_particles, "particles")
mixture.visualize()

Changing the composition is one argument, and the object records what it built.
That record is what makes the study reproducible later.

In [ ]:
for fraction in [0.0, 0.5, 1.0]:
    m = AlkylphenolMixture(4, 12, 40, fraction, 4.5)
    print(f"{fraction:>4} -> {m.composition}")

---

# SMARTS strings and force field XML

## What atom typing actually does

An atom type is not "carbon". It is "carbon in a particular chemical
environment". In MoSDeF, particularly Foyer and GMSO, SMARTS is the pattern language that describes that environment.
foyer converts SMARTS strings to chemical graphs, and walks the molecular graph of an mBuild compound, matching graphs against every atom.

Read these left to right.

| SMARTS | matches |
|---|---|
| `[C;X4]` | carbon with 4 connections |
| `[C;X4](H)(H)(H)O` | sp3 carbon bonded to 3 hydrogens and an oxygen |
| `H[O;X2]` | hydrogen bonded to a 2 connected oxygen |
| `[O;X2](C)H` | 2 connected oxygen bonded to a carbon and a hydrogen |
| `[c;X3]` | aromatic carbon with 3 connections |

In [ ]:
oplsaa = ForceField("oplsaa")

for name in ["opls_145", "opls_146", "opls_148", "opls_166", "opls_167", "opls_168"]:
    at = oplsaa.atom_types[name]
    print(f"{name:<10} {str(at.definition):<44} charge={at.charge}")

Type a few of the molecules from earlier and watch which types appear. The
unsubstituted ring is all `opls_145`. Attach a methyl and its ring carbon
becomes `opls_148`, attach a hydroxyl and the carbon carrying it becomes
`opls_166`.

Watch the three cresols though. They come out identical, because OPLS-AA types
on what a substituent is and not on where it sits around the ring. The isomers
are different molecules with different geometry, and this force field gives them
the same parameters. Knowing where your force field stops resolving chemistry is
part of using it honestly.


In [ ]:
from collections import Counter

for name, (positions, groups) in recipes.items():
    mol = decorate(positions, groups)
    mol.name = name
    top = mol.to_gmso()
    apply(top=top, forcefields=oplsaa, identify_connections=True)
    types = Counter(s.atom_type.name for s in top.sites)
    print(f"{name:<11} {dict(types)}")

## The XML format

A GMSO force field XML has four things in it.

- `FFMetaData` sets the combining rule, the 1-4 scaling factors, and the units
- `AtomTypes` gives each type a name, an atom class, a charge, a SMARTS
  `definition`, and nonbonded parameters
- `BondTypes`, `AngleTypes`, `DihedralTypes` match on atom classes and carry the
  bonded parameters
- Every block declares its own functional form as an `expression` string, so the
  file is self describing

Open `files/methanol.xml` and read it alongside the output below. It is a small
complete force field for one molecule.

In [ ]:
!head -n 40 files/methanol.xml

In [ ]:
methanol_ff = ForceField("files/methanol.xml")

print(methanol_ff.name, "version", methanol_ff.version)
print("combining rule:", methanol_ff.combining_rule)
print("units:", methanol_ff.units)
print()
for name, at in methanol_ff.atom_types.items():
    print(f"{name:<10} {str(at.definition):<20} class={at.atomclass:<3} q={at.charge}")
    print(f"{'':<10} {at.parameters}")

Notice that bonded types match on `atomclass`, not on the atom type name. That
is how a force field stays small. Many types share a class, and the bonded
parameters are written once per class combination.

In [ ]:
for name, bt in methanol_ff.bond_types.items():
    print(name, bt.parameters)
print()
for name, at in methanol_ff.angle_types.items():
    print(name, at.parameters)

## Applying the custom XML
The reason for SMARTS-string based atom-typing, and algebraic specification of interactions in the MoSDeF .xml formats is that it allows users to design, use, and share their own force fields. 

A ForceField built from your own file behaves exactly like one that shipped with the package.

In [ ]:
methanol = mb.load("CO", smiles=True)
methanol.name = "methanol"

box = mb.fill_box(compound=methanol, n_compounds=100, box=[3, 3, 3])
top = box.to_gmso()

apply(
    top=top,
    forcefields=methanol_ff,
    identify_connections=True,
    speedup_by_moltag=True,
    fast_copy=False,
)

print("typed:", top.is_typed())
print(top.n_sites, "sites |", top.n_bonds, "bonds |",
      top.n_angles, "angles |", top.n_dihedrals, "dihedrals")

`fast_copy=False` is worth knowing about. With the default `True`, every site
of the same type shares one `AtomType` object, which is fast but means editing
one edits them all. Setting it to `False` gives each site its own copy, which is
what you want if you are about to modify parameters selectively.

In [ ]:
unique = {}
for site in top.sites:
    unique.setdefault(site.atom_type.name, site.atom_type)

for name, at in unique.items():
    print(f"{name:<10} {at.parameters}")

## Editing the force field on a typed Topology

After calling `apply()` on a GMSO Topology, the complete force field parameters for the system are stored in the GMSO Topology Python object.
These parameters are also editable, which is useful for performing tasks such as assigning and/or scaling charges, iterating over modificaitons to epsilon, sigma, force constants, and more.

As an example, this could be used for a sensitivity study, since you apply and atom type once and then perturb. This kind of workflow can be much more manageable than storing and organizing a large set of xml files.

Below we soften the oxygen by shrinking its `sigma` in memory, rather than changing the .xml file. We confirm the change
landed on every oxygen site.

In [ ]:
import unyt as u

before = unique["MeOH_O"].parameters["sigma"]
print("before:", before)

for site in top.sites:
    if site.atom_type.name == "MeOH_O":
        site.atom_type.parameters["sigma"] = 0.290 * u.nm

oxygens = [s for s in top.sites if s.atom_type.name == "MeOH_O"]
after = set(str(s.atom_type.parameters["sigma"]) for s in oxygens)
print("after :", after, f"across {len(oxygens)} sites")

## A simple screening study

Now put that same edit in a loop. For each value of the parameter, pack a fresh
system, type it, edit the typed Topology, and write the engine inputs into their
own directory.

`fast_copy=False` again, for the reason given above. Every site carries its own
`AtomType`, so the edit stays on this topology and does not reach back into the
`ForceField` object the loop keeps reusing.

The writers read the Topology as it stands, and anything they work out for unlike
pairs is worked out there from those values. So the number you set here is the
number that ends up in the inputs.

In [ ]:
from pathlib import Path

sigma_values = [0.290, 0.300, 0.312, 0.325]
runs = []

for sigma in sigma_values:
    system = mb.fill_box(compound=methanol, n_compounds=100, box=[3, 3, 3], seed=42)
    topology = system.to_gmso()
    apply(topology, methanol_ff, identify_connections=True,
          speedup_by_moltag=True, fast_copy=False)

    for site in topology.sites:
        if site.atom_type.name == "MeOH_O":
            site.atom_type.parameters["sigma"] = sigma * u.nm

    outdir = Path(f"screen/sigma_{sigma:.3f}")
    outdir.mkdir(parents=True, exist_ok=True)
    write_gro(topology, str(outdir / "methanol.gro"))
    write_top(topology, str(outdir / "methanol.top"))

    runs.append({"sigma": sigma, "path": outdir})
    print(f"sigma = {sigma:.3f} nm -> {outdir}")

In [ ]:
!find screen -type f | sort

Every directory holds a complete, runnable input set, and the only thing that
differs between them is the one number you varied. The loop is the
documentation.

In [ ]:
for run in runs:
    with open(run["path"] / "methanol.top") as f:
        line = [l for l in f if "MeOH_O" in l][0]
    print(f"{run['sigma']:.3f}  {line.strip()}")

<h1 style="color: green;">Exercise</h1>

Write the atom type definitions for a **united atom** methanol where all Hydrogen atoms are removed.

The all atom model above has four types, the carbon plus its three hydrogens plus
the hydroxyl pair. A united atom model folds a carbon and the hydrogens riding on
it into one site, so methanol drops from six atoms to three sites. Three sites in
a row also means there is no dihedral to write, so the file loses a whole block.

The XML is given below as a template, already carrying the TraPPE-UA parameters.
The three `definition` attributes are the blanks, and they are the part worth
thinking about.

| site | class | element | mass | charge | sigma (nm) | epsilon (kJ/mol) |
|---|---|---|---|---|---|---|
| `CH3` | CH3 | C | 15.035 | +0.265 | 0.375 | 0.814821 |
| `O` | OH | O | 15.999 | -0.700 | 0.302 | 0.773249 |
| `H` | HO | H | 1.008 | +0.435 | 0.100 | 0.0 |

1. Write a SMARTS `definition` for each of the three sites and pass them to
   `write_ua_xml`
2. Build a three site methanol Compound by hand, since a SMILES string only
   knows how to give you atoms
3. Pack 50 of them, apply your force field, and check `is_typed()`, the site
   count, and that the net charge lands on zero
4. Screen the `MeOH_CH3` epsilon over a few values and write LAMMPS data files
   instead of GROMACS ones

Two things worth knowing before you start.

- The methyl bead is still a carbon as far as the force field is concerned. It
  carries `element="C"` and `mass="15.035"`, so the three hydrogens live in the
  mass rather than in the bond graph. That is what makes its SMARTS easy, since
  it leaves the bead as the only carbon in the molecule with a single connection
- Building a particle by hand means passing `element=` explicitly, as in
  `mb.Compound(name="O", element="O")`. mBuild will not infer it from the name
  here, and foyer raises if it is missing

Tip. Get one wrong and you hear about it straight away. Two patterns matching the
same atom raises an ambiguity, and none matching raises a missing type.

In [ ]:
UA_TEMPLATE = """<?xml version='1.0' encoding='UTF-8'?>
<ForceField name="methanol-ua-workshop" version="0.0.1">
  <FFMetaData electrostatics14Scale="0.0" nonBonded14Scale="0.0" combiningRule="lorentz">
    <Units energy="kJ/mol" distance="nm" mass="amu" charge="elementary_charge"/>
  </FFMetaData>
  <AtomTypes expression="4*epsilon*((sigma/r)**12 - (sigma/r)**6)">
    <ParametersUnitDef parameter="epsilon" unit="kJ/mol"/>
    <ParametersUnitDef parameter="sigma" unit="nm"/>
    <AtomType name="MeOH_CH3" atomclass="CH3" element="C" mass="15.035" charge="0.265" definition="{ch3}">
      <Parameters>
        <Parameter name="epsilon" value="0.814821"/>
        <Parameter name="sigma" value="0.375"/>
      </Parameters>
    </AtomType>
    <AtomType name="MeOH_O" atomclass="OH" element="O" mass="15.999" charge="-0.700" definition="{o}">
      <Parameters>
        <Parameter name="epsilon" value="0.773249"/>
        <Parameter name="sigma" value="0.302"/>
      </Parameters>
    </AtomType>
    <AtomType name="MeOH_HO" atomclass="HO" element="H" mass="1.008" charge="0.435" definition="{h}">
      <Parameters>
        <Parameter name="epsilon" value="0.0"/>
        <Parameter name="sigma" value="0.100"/>
      </Parameters>
    </AtomType>
  </AtomTypes>
  <BondTypes expression="0.5*k*(r - r_eq)**2">
    <ParametersUnitDef parameter="k" unit="kJ/(mol*nm**2)"/>
    <ParametersUnitDef parameter="r_eq" unit="nm"/>
    <BondType name="HarmonicBondPotential" type1="CH3" type2="OH">
      <Parameters>
        <Parameter name="k" value="267776.0"/>
        <Parameter name="r_eq" value="0.143"/>
      </Parameters>
    </BondType>
    <BondType name="HarmonicBondPotential" type1="OH" type2="HO">
      <Parameters>
        <Parameter name="k" value="462750.4"/>
        <Parameter name="r_eq" value="0.0945"/>
      </Parameters>
    </BondType>
  </BondTypes>
  <AngleTypes expression="0.5*k*(theta - theta_eq)**2">
    <ParametersUnitDef parameter="k" unit="kJ/(mol*rad**2)"/>
    <ParametersUnitDef parameter="theta_eq" unit="rad"/>
    <AngleType name="HarmonicAnglePotential" type1="CH3" type2="OH" type3="HO">
      <Parameters>
        <Parameter name="k" value="460.620"/>
        <Parameter name="theta_eq" value="1.893682"/>
      </Parameters>
    </AngleType>
  </AngleTypes>
</ForceField>
"""


def write_ua_xml(ch3, o, h, path="files/methanol_ua.xml"):
    """Fill the three SMARTS blanks into the template and load the result."""
    Path(path).write_text(UA_TEMPLATE.format(ch3=ch3, o=o, h=h))
    return ForceField(path)


# the three blanks you need to fill
for line in UA_TEMPLATE.splitlines():
    if "definition=" in line:
        print(line.strip())

In [ ]:
# Your code here

<h2 style="color: blue;">Answer</h2>

Run the cells below to see one solution.

In [ ]:
ua_ff = write_ua_xml(ch3="[C;X1](O)", o="[O;X2](C)H", h="H[O;X2]")

for name, at in ua_ff.atom_types.items():
    print(f"{name:<9} {str(at.definition):<11} class={at.atomclass:<4} q={at.charge} m={at.mass}")

print(len(ua_ff.atom_types), "atom types |", len(ua_ff.bond_types), "bond |",
      len(ua_ff.angle_types), "angle |", len(ua_ff.dihedral_types), "dihedral")

The Compound is built particle by particle. Positions come straight from the
bond lengths and the angle, and the two `add_bond` calls are what make it a
molecule rather than three loose sites.

In [ ]:
ua = mb.Compound(name="methanol")
ua.add(mb.Compound(name="CH3", element="C", pos=[0.000, 0.0000, 0.0]))
ua.add(mb.Compound(name="O", element="O", pos=[0.143, 0.0000, 0.0]))
ua.add(mb.Compound(name="H", element="H", pos=[0.173, 0.0896, 0.0]))
ua.add_bond((ua.children[0], ua.children[1]))
ua.add_bond((ua.children[1], ua.children[2]))

ua_box = mb.fill_box(compound=ua, n_compounds=50, box=[3, 3, 3], seed=42)
ua_top = ua_box.to_gmso()
apply(ua_top, ua_ff, identify_connections=True, speedup_by_moltag=True)

print("typed:", ua_top.is_typed())
print(ua_top.n_sites, "sites |", ua_top.n_bonds, "bonds |",
      ua_top.n_angles, "angles |", ua_top.n_dihedrals, "dihedrals")
print("net charge:", round(sum(float(s.atom_type.charge.value) for s in ua_top.sites), 6))

150 sites where the all atom model gave 600, and no dihedrals at all, which is
most of the reason united atom models are worth the loss of resolution.

The screening loop is the same shape as the one above, editing the typed
Topology and writing LAMMPS instead of GROMACS.

In [ ]:
for epsilon in [0.70, 0.815, 0.95]:
    system = mb.fill_box(compound=ua, n_compounds=50, box=[3, 3, 3], seed=42)
    topology = system.to_gmso()
    apply(topology, ua_ff, identify_connections=True,
          speedup_by_moltag=True, fast_copy=False)

    for site in topology.sites:
        if site.atom_type.name == "MeOH_CH3":
            site.atom_type.parameters["epsilon"] = epsilon * u.Unit("kJ/mol")

    outdir = Path(f"screen_ua/eps_{epsilon:.3f}")
    outdir.mkdir(parents=True, exist_ok=True)
    write_lammpsdata(topology, str(outdir / "methanol_ua.data"),
                     atom_style="full", unit_style="real")
    print(f"epsilon = {epsilon:.3f} kJ/mol -> {outdir}")

---

### Recap

- Ports plus `force_overlap` let you snap Compounds together, and popping a
  hydrogen off any molecule turns it into a piece
- Subclassing `Compound` turns a build into a reusable, parameterized object.
  A core and a substituent compose without knowing about each other
- A recipe class records what it built, which is what makes a study reproducible
- SMARTS `definition` strings are how a force field says what an atom type means,
  which is why substituting a ring changes the types on its carbons
- A force field XML declares its own functional forms, so it is readable on its own
- Editing a `ForceField` before `apply` is the clean way to run a parameter sweep

Next up is the polymer notebook, where the structures get a lot more
interesting.